# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`

This notebook demonstrates how to programmatically load, explore, and process the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library and Croissant schemas.

### Dataset Source
The dataset is described via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Display main metadata fields
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")
print(f"Authors: {getattr(dataset.metadata, 'author', 'N/A')}")


## 2. Data Overview
Let's review the available record sets and their Croissant `@id`s, and list the fields within each record set.

In Croissant, each logical table or dataset component is called a **record set** and referenced via its unique `@id`. Each record set contains **fields** (like columns), also uniquely identified via `@id`.

In [ ]:
# List all record sets and their fields using their @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset schema.")
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']} ({rs.get('name','')})")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for f in fields:
            # Either a field object or @id reference
            if isinstance(f, dict):
                field_id = f.get('@id', 'MISSING_FIELD_ID')
                field_name = f.get('name', '')
            else:
                field_id = f
                field_name = ''
            print(f"  - Field: {field_id} {('('+ field_name + ')') if field_name else ''}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> **Note:** Use the record set `@id` and field `@id`s printed above. If `record_sets` is empty or not populated, you may need to try known `@id`s or check the dataset's data files individually.

In [ ]:
# For this FAIR² schema, available record sets are referenced by their `@id`.
# Example pattern for extracting all record sets if available:

all_record_sets = dataset.record_sets
record_set_ids = [rs['@id'] for rs in all_record_sets] if all_record_sets else []

if not record_set_ids:
    print("No record sets were found in the metadata. Attempting to load records using common or known record set IDs.")
    # If there's no record_set list, often the @id is the main dataset URL + '/records'
    # Or consult the data file directly
    # For illustration, use a placeholder (you may need to update this to the correct @id):
    record_set_ids = ['http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8336ac61-9308-403f-8df3-28e120cc98f3']

dataframes = {}
for rsid in record_set_ids:
    # Attempt to extract records for each record set @id
    print(f"\nLoading records for record set: {rsid}")
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            print(f"Loaded {len(df)} records, columns: {df.columns.tolist()}")
            dataframes[rsid] = df
        else:
            print("  No records extracted (empty).")
    except Exception as e:
        print(f"  Failed to load: {e}")

# Example: print the first few rows of the first available dataframe
if dataframes:
    first_rsid = next(iter(dataframes))
    print(f"\nColumns in record set {first_rsid}:\n{list(dataframes[first_rsid].columns)}\n")
    display(dataframes[first_rsid].head())
else:
    print("No dataframes were successfully loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's process numeric fields, filter records, and compute normalized values or grouped statistics.

> **Tips:** Use the field `@id` from above (in practice, this is usually the column name in the DataFrame). If available, choose a numeric field, e.g., `'log_likelihood'`, `'coefficient'`, or another result variable, and a categorical/grouping field such as `'variable'`, `'region'`, or similar.

In [ ]:
# You may need to adjust the field @id here based on the columns present in the loaded DataFrame.
# For this example, we'll look for 'log_likelihood' and 'variable' fields.

import numpy as np

if dataframes:
    first_rsid = next(iter(dataframes))
    df = dataframes[first_rsid]
    # Try to infer possible numeric/categorical fields
    candidate_numeric_fields = [c for c in df.columns if any(s in c.lower() for s in ['log_likelihood', 'coef', 'std_err', 'pvalue', 'score', 'value', 'rate'])]
    group_fields = [c for c in df.columns if any(s in c.lower() for s in ['group', 'variable', 'region', 'ward', 'category', 'gender'])]

    print(f"Numeric candidates: {candidate_numeric_fields}")
    print(f"Group candidates: {group_fields}")
    
    # Pick the first candidate for illustration
    numeric_field_id = candidate_numeric_fields[0] if candidate_numeric_fields else None
    group_field = group_fields[0] if group_fields else None

    if numeric_field_id:
        # Ensure numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanmean(df[numeric_field_id])

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by group_field
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            print(f"Grouped mean by '{group_field}':")
            display(grouped_df)
    else:
        print("No obvious numeric field detected for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize the distribution of a chosen numeric field, and its relationship to a grouping variable (if available).

> Adjust the fields to match what's present in your DataFrame. The plot will only work if numeric and group columns exist.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
- This notebook demonstrated how to load and process a FAIR² (Croissant-schema) dataset using `mlcroissant`, referencing all entities by their `@id` fields.
- We listed available record sets, extracted data to DataFrames, identified key numeric fields, and visualized their distribution.
- This workflow serves as a reproducible template for exploring any Croissant-packaged dataset with record, field, and column `@id` references.

> **Next steps**: You can further analyze the dataset by applying domain-specific filters, training statistical models, or joining with related resources, always referencing entities by their schema `@id` for clarity and reproducibility.